# Missing Indicators (Binary Flags)

A **missing indicator** is a binary feature ($1$ or $0$) that explicitly flags whether a value was originally missing in an observation.

When paired with mean, median, or arbitrary value imputation, a missing indicator allows the machine learning algorithm to distinguish between truly measured values and imputed estimates.

---

## Learning Objectives
By the end of this notebook, you will be able to:
1. Understand why pairing imputation with a missing indicator preserves predictive signals (especially under MAR / MNAR mechanisms).
2. Avoid the common anti-pattern of creating redundant indicators for columns with zero missing values.
3. Prevent data leakage by learning which variables require indicators strictly from the training set.
4. Implement missing indicators using **pandas**, **scikit-learn's `MissingIndicator`**, and **`SimpleImputer(add_indicator=True)`**.

---
## 1. Why Use Missing Indicators?

When we impute missing values (e.g., replacing missing `LotFrontage` with the median $69$), the model cannot tell whether a property truly has a lot frontage of 69 feet, or if the data was missing.

If the fact that data is missing carries predictive information (e.g., patients who did not take a test are healthier, or houses without frontage records differ systematically):
- The **imputed column** provides a plausible numerical value so the model can compute matrix multiplications.
- The **indicator column** ($LotFrontage\_NA \in \{0, 1\}$) preserves the information that the value was imputed.

### Best Practice: Targeted Indicators vs. Naive Indicators
- **Naive approach**: Adding an indicator column for every feature in the dataset. This creates dozens of all-zero columns for complete features, bloating the dataset and wasting compute.
- **Best Practice**: Only create indicators for variables that actually have missing values in the training set.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, MissingIndicator

pd.set_option('display.max_columns', None)

---
## 2. Loading Data & Splitting Train/Test Sets

We use numerical and categorical features from the Ames House Prices dataset.

In [2]:
cols_to_use = [
    "OverallQual",
    "TotalBsmtSF",
    "1stFlrSF",
    "GrLivArea",
    "LotFrontage",
    "MasVnrArea",
    "GarageYrBlt",
    "SalePrice",
]

data_path = '../datasets/houseprice.csv'
if not os.path.exists(data_path):
    data_path = 'https://raw.githubusercontent.com/cmps460s26/cmps460-content/refs/heads/main/examples/03.dp/datasets/houseprice.csv'

data = pd.read_csv(data_path, usecols=cols_to_use)

# Separate predictors and target
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]

# Train/test split BEFORE any transformation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (1022, 7)
X_test shape:  (438, 7)


---
## 3. Identifying Features With Missing Values

We identify which variables contain missing values **in the training set only**.

In [3]:
# Find variables with missing values in X_train
vars_with_na = [col for col in X_train.columns if X_train[col].isnull().sum() > 0]

print("Variables requiring missing indicators:")
for col in vars_with_na:
    pct = X_train[col].isnull().mean() * 100
    print(f"- {col:15s}: {pct:.2f}% missing ({X_train[col].isnull().sum()} rows)")

Variables requiring missing indicators:
- LotFrontage    : 18.49% missing (189 rows)
- MasVnrArea     : 0.49% missing (5 rows)
- GarageYrBlt    : 5.28% missing (54 rows)


---
## 4. Method 1: Targeted Indicators with Pandas

We create binary indicator columns named `{col}_na` for the targeted variables, then impute the original variables with their median.

In [4]:
# Create explicit copies
X_train_pd = X_train.copy()
X_test_pd = X_test.copy()

# Add binary indicators for variables with missing values
for col in vars_with_na:
    indicator_name = f"{col}_na"
    X_train_pd[indicator_name] = X_train_pd[col].isnull().astype(int)
    X_test_pd[indicator_name] = X_test_pd[col].isnull().astype(int)

# Learn median values from training set
imputation_dict = X_train[vars_with_na].median().to_dict()

# Impute the original columns with the median
X_train_pd.fillna(imputation_dict, inplace=True)
X_test_pd.fillna(imputation_dict, inplace=True)

print("Sample of transformed training data (original imputed feature + indicator):")
sample_cols = [c for c in X_train_pd.columns if any(v in c for v in vars_with_na)]
X_train_pd[sample_cols].head()

Sample of transformed training data (original imputed feature + indicator):


,LotFrontage,MasVnrArea,GarageYrBlt,LotFrontage_na,MasVnrArea_na,GarageYrBlt_na
64,69.0,573.0,1998.0,1,0,0
682,69.0,0.0,1996.0,1,0,0
960,50.0,0.0,1979.0,0,0,1
1384,60.0,0.0,1939.0,0,0,0
1100,60.0,0.0,1930.0,0,0,0


In [5]:
# Verify that indicators accurately flag the imputed values
print("Indicator counts in X_train:")
for col in vars_with_na:
    print(f"- {col}_na positive flags: {X_train_pd[f'{col}_na'].sum()}")

Indicator counts in X_train:
- LotFrontage_na positive flags: 189
- MasVnrArea_na positive flags: 5
- GarageYrBlt_na positive flags: 54


---
## 5. Method 2: Scikit-Learn `MissingIndicator`

Scikit-learn provides `MissingIndicator(features='missing-only')` which automatically discovers features with missing values in `fit()` and generates the binary indicator matrix.

In [6]:
# Set up MissingIndicator to target missing features only
indicator = MissingIndicator(features='missing-only')
indicator.set_output(transform="pandas")

# Fit on training data (learns which columns contain NaNs)
indicator.fit(X_train)

# Transform train and test
X_train_ind = indicator.transform(X_train)
X_test_ind = indicator.transform(X_test)

print("Columns generated by MissingIndicator:")
print(X_train_ind.columns.tolist())
X_train_ind.head()

Columns generated by MissingIndicator:
['missingindicator_LotFrontage', 'missingindicator_MasVnrArea', 'missingindicator_GarageYrBlt']


,missingindicator_LotFrontage,missingindicator_MasVnrArea,missingindicator_GarageYrBlt
64,True,False,False
682,True,False,False
960,False,False,True
1384,False,False,False
1100,False,False,False


---
## 6. Method 3: All-In-One Imputation with `SimpleImputer(add_indicator=True)`

Scikit-learn's `SimpleImputer` offers an elegant built-in parameter: `add_indicator=True`.

When set, `SimpleImputer` automatically:
1. Imputes the numerical features using the specified strategy (e.g. median).
2. Appends missing indicator columns for any features that had missing values during `.fit()`.

In [7]:
# Initialize SimpleImputer with median strategy and add_indicator=True
auto_imputer = SimpleImputer(strategy='median', add_indicator=True)
auto_imputer.set_output(transform="pandas")

# Fit on training data and transform both sets
auto_imputer.fit(X_train)

X_train_auto = auto_imputer.transform(X_train)
X_test_auto = auto_imputer.transform(X_test)

print(f"Original feature count:    {X_train.shape[1]}")
print(f"Transformed feature count: {X_train_auto.shape[1]} (original features + indicator columns)")
print(f"Remaining nulls:           {X_train_auto.isnull().sum().sum()}")
X_train_auto.head()

Original feature count:    7
Transformed feature count: 10 (original features + indicator columns)
Remaining nulls:           0


,LotFrontage,OverallQual,MasVnrArea,TotalBsmtSF,1stFlrSF,GrLivArea,GarageYrBlt,missingindicator_LotFrontage,missingindicator_MasVnrArea,missingindicator_GarageYrBlt
64,69.0,7.0,573.0,1057.0,1057.0,2034.0,1998.0,1.0,0.0,0.0
682,69.0,6.0,0.0,1291.0,1291.0,1291.0,1996.0,1.0,0.0,0.0
960,50.0,5.0,0.0,858.0,858.0,858.0,1979.0,0.0,0.0,1.0
1384,60.0,6.0,0.0,560.0,698.0,1258.0,1939.0,0.0,0.0,0.0
1100,60.0,2.0,0.0,290.0,438.0,438.0,1930.0,0.0,0.0,0.0


---
## 7. Summary & Best Practices Checklist

| Consideration | Recommendation |
| :--- | :--- |
| **When to Use** | When missingness is moderate-to-high and may carry predictive meaning (MAR / MNAR). |
| **Avoid Redundant Columns** | Never add indicators to features with $0$ missing values. Use `features='missing-only'` or pandas filtering. |
| **Data Leakage** | The set of columns receiving indicators must be determined strictly from `X_train`. |
| **All-in-One Solution** | Use `SimpleImputer(strategy='median', add_indicator=True)` inside a scikit-learn `Pipeline`. |
